In [6]:
%pip install --upgrade pip        


Note: you may need to restart the kernel to use updated packages.


In [26]:
import pandas as pd
from pathlib import Path

csv_path = Path("data/2005-23-uk-local-authority-ghg-emissions-CSV-dataset.csv")

In [28]:
usecols = ["ladcode",            # Local Authority District 代码
           "year",
           "sector",             # 'Domestic' / 'Industrial' / …
           "ghg",                # 'CO2', 'CH4'…
           "territorial_emission(kt CO2e)",
           "mid-year-population(1000)"]

co2 = (pd.read_csv(csv_path, usecols=usecols)
         .query("sector == 'Domestic' & ghg == 'CO2' & year >= 2008")   # 年份 & 部门筛选
         .rename(columns={"territorial_emission(kt CO2e)": "kt_co2"})   # 易读列名
         .assign(t_co2=lambda x: x.kt_co2 * 1_000)                      # kt→t
         .sort_values(["ladcode", "year"])
)

print(co2.head(10))

       ladcode  year    sector  ghg      kt_co2  mid-year-population(1000)  \
267  E06000001  2008  Domestic  CO2   79.691064                     91.379   
270  E06000001  2008  Domestic  CO2  122.348690                     91.379   
273  E06000001  2008  Domestic  CO2    8.100586                     91.379   
350  E06000001  2009  Domestic  CO2   72.958704                     91.530   
353  E06000001  2009  Domestic  CO2  110.804042                     91.530   
356  E06000001  2009  Domestic  CO2    7.407651                     91.530   
431  E06000001  2010  Domestic  CO2   75.057269                     91.773   
434  E06000001  2010  Domestic  CO2  121.896449                     91.773   
437  E06000001  2010  Domestic  CO2    7.882286                     91.773   
512  E06000001  2011  Domestic  CO2   72.137512                     92.088   

             t_co2  
267   79691.064300  
270  122348.690000  
273    8100.586024  
350   72958.704090  
353  110804.041600  
356    7407.651

In [38]:
co2_agg = (
    co2               
    .groupby(['ladcode', 'year'], as_index=False)
    .agg({
        't_co2': 'sum',                       
        'mid-year-population(1000)': 'first' 
    })
    .rename(columns={'mid-year-population(1000)': 'pop_1000'})
)

# === 2. calculate CO2 emission per 1000
co2_agg['t_co2'] = co2_agg['t_co2'].astype(float).round(3)
co2_agg['co2_per_1000p'] = (co2_agg['t_co2'] / co2_agg['pop_1000']).round(3)

print(co2_agg.head())
co2_agg.to_csv("data/co2_domestic_2008_23_lad.csv", index=False)

     ladcode  year       t_co2  pop_1000  co2_per_1000p
0  E06000001  2008  210140.340    91.379       2299.657
1  E06000001  2009  191170.397    91.530       2088.609
2  E06000001  2010  204836.004    91.773       2231.985
3  E06000001  2011  179926.089    92.088       1953.849
4  E06000001  2012  191699.606    92.344       2075.929


In [34]:
# === 3. input the EPC D1 from https://www.gov.uk/government/statistical-data-sets/live-tables-on-energy-performance-of-buildings-certificates
# === clean out EPC for panel: (Domestic Properties) quarterly CSV -> LAD×year EPC band shares

import pandas as pd
from pathlib import Path
import re

# ========= Config =========
INPUT_CSV  = Path("data/D1_Domestic_Properties.csv")         # 改成你的路径
OUTPUT_DIR = Path("data/epc")                           # 输出目录
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_CSV    = OUTPUT_DIR / "epc_bands_D1_lad_year.csv"
OUT_PQ     = OUTPUT_DIR / "epc_bands_D1_lad_year.parquet"

d1 = pd.read_csv(INPUT_CSV, low_memory=False)
required = ["ladcode", "Quarter", "NumberofLodgements", "A", "B", "C", "D", "E", "F", "G"]
missing = [c for c in required if c not in d1.columns]
if missing:
    raise ValueError(f"缺少关键列：{missing}\n实际列名：{list(d1.columns)}")

In [37]:

# only keep real LAD rows（E/W/S/N + 8位）
d1 = d1[d1["ladcode"].astype(str).str.match(r"^[EWSN]\d{8}$", na=False)].copy()

# ========= 2) To numeric =========
num_cols = ["NumberofLodgements", "A", "B", "C", "D", "E", "F", "G"]
d1[num_cols] = d1[num_cols].apply(pd.to_numeric, errors="coerce").fillna(0)

# extract year from quarter
d1["year"] = d1["Quarter"].astype(str).str.extract(r"(\d{4})").astype(int)

# ========= 3) aggregate to LAD×year =========
agg_dict = {c: "sum" for c in num_cols}
annual = d1.groupby(["ladcode", "year"], as_index=False).agg(agg_dict)

# summarise all properties, get rid of " not recorded"
annual["den_classified"] = annual[["A","B","C","D","E","F","G"]].sum(axis=1)
annual.loc[annual["den_classified"] == 0, "den_classified"] = pd.NA  # avoid denominator is 0


# ========= 4) calculate percantage =========
for b in list("ABCDEFG"):
    annual[f"pct_{b}"] = (100 * annual[b] / annual["den_classified"]).astype(float)

annual["pct_EFG"] = annual[["pct_E", "pct_F", "pct_G"]].sum(axis=1)

pct_cols = [f"pct_{b}" for b in "ABCDEFG"] + ["pct_EFG"]
annual[pct_cols] = annual[pct_cols].astype(float).round(3)

# ========= 5) export and save to data "only EFG"=========
epc_bands = annual[["ladcode","year"] + [f"pct_{b}" for b in "ABCDEFG"]].copy()
epc_bands["pct_EFG"]  = epc_bands[["pct_E","pct_F","pct_G"]].sum(axis=1)
epc_bands["pct_ABCD"] = epc_bands[["pct_A","pct_B","pct_C","pct_D"]].sum(axis=1)

epc_bands.to_csv(OUT_CSV, index=False)
try:
    epc_bands.to_parquet(OUT_PQ, index=False)
except Exception:
    pass

print("✅ Done.")
print(f"Saved CSV: {OUT_CSV}")
print(f"Rows: {len(epc_bands)}")
print(epc_bands.head())


✅ Done.
Saved CSV: data/epc/epc_bands_D1_lad_year.csv
Rows: 6048
     ladcode  year  pct_A   pct_B   pct_C   pct_D   pct_E  pct_F  pct_G  \
0  E06000001  2008  0.000  12.361  26.180  41.116  15.536  4.292  0.515   
1  E06000001  2009  0.030  16.888  29.023  36.168  13.847  3.159  0.886   
2  E06000001  2010  0.176  11.272  30.116  41.247  12.786  3.346  1.057   
3  E06000001  2011  0.620   9.179  27.808  44.152  14.601  2.672  0.968   
4  E06000001  2012  0.040   4.928  30.248  48.638  13.141  2.444  0.561   

   pct_EFG  pct_ABCD  
0   20.343    79.657  
1   17.892    82.109  
2   17.189    82.811  
3   18.241    81.759  
4   16.146    83.854  


In [18]:
import pandas as pd
import numpy as np
import re
import warnings
from pathlib import Path
import statsmodels.formula.api as smf

EPC_PATH = "data/epc/epc_bands_D1_lad_year.csv"      # ladcode, year, pct_EFG（若无会尝试从 pct_E+F+G 计算）
CO2_PATH = "data/co2_domestic_2008_23_lad.csv"   # ladcode, year, t_co2；建议含 pop_1000 或 co2_per_1000p
rural_PATH  = "data/rural-urban-lad.csv" # 官方 LAD rural/urban 分类（任一列包含 urban/rural 关键词）

OUT_RESULTS = "data/rq1_results.csv"
OUT_PANEL   = "data/rq1_merged_panel.csv"

OUT_EPC = "data/epc_for_regression.csv"
OUT_CO2 = "data/co2_for_regression.csv"
OUT_RU = "data/ru_for_regression.csv"

In [13]:
# -------------------------------
# 1) read data and clean 
# -------------------------------
epc = pd.read_csv(EPC_PATH)
co2 = pd.read_csv(CO2_PATH)
ru  = pd.read_csv(rural_PATH)

def std_keys(df):
    if "ladcode" in df.columns:
        df["ladcode"] = df["ladcode"].astype(str).str.strip().str.upper()
    if "year" in df.columns:
        df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")
    return df

ru = ru.rename(columns={"LAD24CD": "ladcode",
                       "LAD24NM": "ladname"})

epc, co2, ru = (std_keys(x) for x in (epc, co2, ru))
epc = epc.drop_duplicates(subset=["ladcode","year"]).copy()
co2 = co2.drop_duplicates(subset=["ladcode","year"]).copy()

print(ru.head())

     ladcode               ladname RUC21CD  \
0  E06000001            Hartlepool     UUN   
1  E06000002         Middlesbrough     UUN   
2  E06000003  Redcar and Cleveland     UIN   
3  E06000004      Stockton-on-Tees     UUN   
4  E06000005            Darlington     UUN   

                                             RUC21NM Rural Urban flag  \
0     Urban: Majority nearer to a major town or city            Urban   
1     Urban: Majority nearer to a major town or city            Urban   
2  Intermediate urban: Majority nearer to a major...            Urban   
3     Urban: Majority nearer to a major town or city            Urban   
4     Urban: Majority nearer to a major town or city            Urban   

  RUC21 settlement class  Proportion of population in rural OAs (%)  \
0                  Urban                                        3.1   
1                  Urban                                        0.2   
2     Intermediate urban                                       31.7   


In [21]:
# 'Rural Urban flag' -> urbanity (Urban=1, Rural=0)
flag_col = next(c for c in ru.columns if c.strip().lower() == "rural urban flag")
ru["urbanity"] = ru[flag_col].astype(str).str.strip().str.contains("urban", case=False, na=False).astype("int8")

print(ru[["ladcode", flag_col, "urbanity"]].head())
print(ru["urbanity"].value_counts(dropna=False))

ru.to_csv(OUT_RU, index=False)
co2.to_csv(OUT_CO2, index=False)
epc.to_csv(OUT_EPC, index=False)

     ladcode Rural Urban flag  urbanity
0  E06000001            Urban         1
1  E06000002            Urban         1
2  E06000003            Urban         1
3  E06000004            Urban         1
4  E06000005            Urban         1
urbanity
1    231
0     87
Name: count, dtype: int64


In [23]:
co2 = co2.sort_values(["ladcode","year"]).drop_duplicates(["ladcode","year"], keep="last")
epc = epc.sort_values(["ladcode","year"]).drop_duplicates(["ladcode","year"], keep="last")
ru  = ru.sort_values(["ladcode"]).drop_duplicates(["ladcode"], keep="last")

In [25]:

base = Path("data")

panel = (co2.merge(epc, on=["ladcode","year"], how="inner", validate="one_to_one")
              .merge(ru[["ladcode","urbanity"]], on="ladcode", how="left", validate="many_to_one"))

# 友好排列（把核心变量放到前面）
cols_front = ["ladcode","year","urbanity","co2_per_1000p","t_co2","pop_1000",
              "pct_E","pct_F","pct_G","pct_EFG","pct_ABCD","pct_A","pct_B","pct_C","pct_D"]
panel = panel[[c for c in cols_front if c in panel.columns] + [c for c in panel.columns if c not in cols_front]]

# 保存（容器里没装 pyarrow，就先存 CSV / pkl）
panel.to_csv(base / "rq1_panel.csv", index=False)
panel.to_pickle(base / "rq1_panel.pkl")


panel.shape, panel["year"].min(), panel["year"].max(), panel["urbanity"].isna().sum()


((4990, 15), 2008, 2023, 0)

In [27]:
import numpy as np
import statsmodels.api as sm

# 清掉关键列缺失
panel = panel.dropna(subset=["co2_per_1000p", "pct_EFG", "ladcode", "year", "urbanity"])

# 明确转成 numpy 整型
panel["year"] = panel["year"].astype("int64")
panel["urbanity"] = panel["urbanity"].astype("int64")  # 若也显示为 Int64 就一并转

import statsmodels.formula.api as smf
m1 = smf.ols("co2_per_1000p ~ pct_EFG + C(ladcode) + C(year)", data=panel) \
        .fit(cov_type="cluster", cov_kwds={"groups": panel["ladcode"]})

print(m1.summary())



                            OLS Regression Results                            
Dep. Variable:          co2_per_1000p   R-squared:                       0.989
Model:                            OLS   Adj. R-squared:                  0.989
Method:                 Least Squares   F-statistic:                 2.475e+04
Date:                Fri, 08 Aug 2025   Prob (F-statistic):               0.00
Time:                        13:24:36   Log-Likelihood:                -26250.
No. Observations:                4990   AIC:                         5.317e+04
Df Residuals:                    4656   BIC:                         5.534e+04
Df Model:                         333                                         
Covariance Type:              cluster                                         
                              coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------
Intercept                2

/opt/conda/lib/python3.11/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 333, but rank is 16
  warnings.warn('covariance of constraints does not have full '


In [28]:
b  = m1.params["pct_EFG"]
se = m1.bse["pct_EFG"]
lo, hi = m1.conf_int().loc["pct_EFG"]
print(f"[m1] β(1pp EFG) = {b:.3f}  (se={se:.3f}, 95%CI=[{lo:.3f},{hi:.3f}])")
print(f"    → 10pp 变化的效应：{10*b:.2f}")

[m1] β(1pp EFG) = -0.853  (se=0.224, 95%CI=[-1.292,-0.414])
    → 10pp 变化的效应：-8.53


In [29]:
# 2) 城市 vs 郊区异质性（m2）
m2 = smf.ols("co2_per_1000p ~ pct_EFG*urbanity + C(ladcode) + C(year)", data=panel) \
        .fit(cov_type="cluster", cov_kwds={"groups": panel["ladcode"]})

V = m2.cov_params(); p = m2.params
beta_rural = p["pct_EFG"]
se_rural   = np.sqrt(V.loc["pct_EFG","pct_EFG"])
beta_city  = p["pct_EFG"] + p["pct_EFG:urbanity"]
se_city    = np.sqrt(V.loc["pct_EFG","pct_EFG"] +
                     V.loc["pct_EFG:urbanity","pct_EFG:urbanity"] +
                     2*V.loc["pct_EFG","pct_EFG:urbanity"])
print(f"[m2] Rural/Suburban: β={beta_rural:.3f} (se={se_rural:.3f})")
print(f"[m2] Urban:          β={beta_city:.3f}  (se={se_city:.3f})")

[m2] Rural/Suburban: β=2.275 (se=0.595)
[m2] Urban:          β=-1.463  (se=0.275)


In [31]:
# 4) 拟合：WLS + 聚类稳健 SE（按 LAD）
# 确保类型/缺失值
need = ["co2_per_1000p","pct_EFG","ladcode","year","pop_1000"]
panel = panel.dropna(subset=need).copy()
panel["year"] = panel["year"].astype("int64")
panel["ladcode"] = panel["ladcode"].astype(str)

# m4：人口加权 WLS + 聚类稳健SE
m4 = smf.wls(
    "co2_per_1000p ~ pct_EFG + C(ladcode) + C(year)",
    data=panel,
    weights=panel["pop_1000"] * 1000
).fit(cov_type="cluster", cov_kwds={"groups": panel["ladcode"]})

# 取 β 和 se
beta = float(m4.params["pct_EFG"])
se   = float(m4.bse["pct_EFG"])
lo, hi = m4.conf_int().loc["pct_EFG"]

print(f"[m4] β(+1pp EFG)={beta:.3f}  se={se:.3f}  95%CI=[{lo:.3f},{hi:.3f}]  N={int(m4.nobs)}")

[m4] β(+1pp EFG)=-0.350  se=0.169  95%CI=[-0.682,-0.019]  N=4990


In [32]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

# —— 清洗与类型 ——
panel = panel.dropna(subset=["co2_per_1000p", "pct_EFG", "ladcode", "year", "urbanity"]).copy()
panel["year"] = panel["year"].astype("int64")
panel["urbanity"] = panel["urbanity"].astype("int64")
# 可选：把 LAD 设成分类，内存友好
panel["ladcode"] = panel["ladcode"].astype("category")

# —— 基线 OLS + FE（用虚拟变量吸收 LAD/Year）——
m1 = smf.ols("co2_per_1000p ~ pct_EFG + C(ladcode) + C(year)", data=panel) \
        .fit(cov_type="cluster", cov_kwds={"groups": panel["ladcode"]})

# 完整回归表（含 p 值、R² 等）
print(m1.summary())

# 核心系数与区间（保留三位小数）
b  = m1.params["pct_EFG"]
se = m1.bse["pct_EFG"]
lo, hi = m1.conf_int().loc["pct_EFG"]
pval = m1.pvalues["pct_EFG"]
print(f"[m1] β(1pp EFG) = {b:.3f}  (se={se:.3f}, p={pval:.3f}, 95%CI=[{lo:.3f},{hi:.3f}])")
print(f"     → 10pp 变化的效应：{10*b:.3f} 吨 CO₂ / 1000人")

# —— Overall R²（来自 statsmodels）——
print(f"R² overall = {m1.rsquared:.3f}   R² adj = {m1.rsquared_adj:.3f}")

# —— Within R²（固定效应“组内”R²，更该看的指标）——
# 通过双向去均值（吸收 LAD 与 Year）来计算
y = panel["co2_per_1000p"]
x = panel["pct_EFG"]
g = panel["ladcode"]
t = panel["year"]

y_dm = y - y.groupby(g).transform("mean") - y.groupby(t).transform("mean") + y.mean()
x_dm = x - x.groupby(g).transform("mean") - x.groupby(t).transform("mean") + x.mean()

m_within = sm.OLS(y_dm, sm.add_constant(x_dm)).fit(
    cov_type="cluster", cov_kwds={"groups": panel["ladcode"]}
)
print(f"R² within (FE) = {m_within.rsquared:.3f}")

# —— 整体显著性（F 统计量 p 值）——
if hasattr(m1, "f_pvalue"):
    print(f"F-test p-value (overall): {m1.f_pvalue:.3f}")

# —— 可选：用 linearmodels 跑 Driscoll–Kraay 稳健协方差作对照 —— 
try:
    from linearmodels.panel import PanelOLS
    df_fe = panel.set_index(["ladcode", "year"]).sort_index()
    res_dk = PanelOLS.from_formula(
        "co2_per_1000p ~ 1 + pct_EFG + EntityEffects + TimeEffects", data=df_fe
    ).fit(cov_type="kernel")  # Driscoll–Kraay 风格
    print(res_dk.summary)
except Exception as e:
    print("（可选）linearmodels 未安装或不可用，跳过 Driscoll–Kraay 对照。")


                            OLS Regression Results                            
Dep. Variable:          co2_per_1000p   R-squared:                       0.989
Model:                            OLS   Adj. R-squared:                  0.989
Method:                 Least Squares   F-statistic:                 2.475e+04
Date:                Fri, 08 Aug 2025   Prob (F-statistic):               0.00
Time:                        16:58:50   Log-Likelihood:                -26250.
No. Observations:                4990   AIC:                         5.317e+04
Df Residuals:                    4656   BIC:                         5.534e+04
Df Model:                         333                                         
Covariance Type:              cluster                                         
                              coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------
Intercept                2

/opt/conda/lib/python3.11/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 333, but rank is 16
  warnings.warn('covariance of constraints does not have full '
/tmp/ipykernel_121/4021296137.py:38: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  y_dm = y - y.groupby(g).transform("mean") - y.groupby(t).transform("mean") + y.mean()
/tmp/ipykernel_121/4021296137.py:39: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  x_dm = x - x.groupby(g).transform("mean") - x.groupby(t).transform("mean") + x.mean()


In [33]:
# ---------- 0) 数据准备 ----------
# 要求 panel 至少包含：co2_per_1000p, pct_F, pct_G, ladcode, year
req = {"co2_per_1000p", "pct_F", "pct_G", "ladcode", "year"}
missing = req - set(panel.columns)
if missing:
    raise ValueError(f"缺少列：{missing}")

df = panel.copy()
df = df.dropna(subset=list(req))
df["year"] = df["year"].astype("int64")
df["ladcode"] = df["ladcode"].astype("category")

# 生成 FG 合并指标（单位依然是“百分点”）
df["pct_FG"] = df["pct_F"].astype(float) + df["pct_G"].astype(float)

# ---------- 一个小工具：计算 within R² ----------
def within_r2(y, X, unit, time):
    """
    y: pd.Series 因变量
    X: pd.Series 自变量（一个列）
    unit, time: 分别是 LAD 和年份的列名
    返回： within R²（FE 组内 R²）
    """
    y = y.copy()
    x = X.copy()
    g = df[unit]
    t = df[time]

    y_dm = y - y.groupby(g).transform("mean") - y.groupby(t).transform("mean") + y.mean()
    x_dm = x - x.groupby(g).transform("mean") - x.groupby(t).transform("mean") + x.mean()
    m = sm.OLS(y_dm, sm.add_constant(x_dm)).fit()
    return float(m.rsquared)

# ---------- 1) 分档模型：%F 与 %G 同时进入（以 D 档为基准的另一种思路是逐档，这里按你要求仅放 F/G） ----------
m_fg_sep = smf.ols(
    "co2_per_1000p ~ pct_F + pct_G + C(ladcode) + C(year)", data=df
).fit(cov_type="cluster", cov_kwds={"groups": df["ladcode"]})

print("\n=== Model A1: Separate %F and %G (FE: LAD + Year, cluster by LAD) ===")
print(m_fg_sep.summary())

# 取关键信息（四舍五入到 3 位）
for var in ["pct_F", "pct_G"]:
    b  = m_fg_sep.params[var]
    se = m_fg_sep.bse[var]
    p  = m_fg_sep.pvalues[var]
    lo, hi = m_fg_sep.conf_int().loc[var]
    print(f"{var}: beta={b:.3f}, se={se:.3f}, p={p:.3f}, 95%CI=[{lo:.3f},{hi:.3f}]  -> 10pp effect={10*b:.3f}")

print(f"R² overall = {m_fg_sep.rsquared:.3f} | R² adj = {m_fg_sep.rsquared_adj:.3f}")

# 用合成指标 %FG 计算 within R²（若想看 %F 或 %G 的 within R²，替换第二个参数）
r2_within_fg = within_r2(df["co2_per_1000p"], df["pct_FG"], "ladcode", "year")
print(f"R² within (using %FG as single regressor for within-R² calc) = {r2_within_fg:.3f}")

# ---------- 2) 合并模型：%FG ----------
m_fg = smf.ols(
    "co2_per_1000p ~ pct_FG + C(ladcode) + C(year)", data=df
).fit(cov_type="cluster", cov_kwds={"groups": df["ladcode"]})

print("\n=== Model A2: Combined %FG (FE: LAD + Year, cluster by LAD) ===")
print(m_fg.summary())

b  = m_fg.params["pct_FG"]
se = m_fg.bse["pct_FG"]
p  = m_fg.pvalues["pct_FG"]
lo, hi = m_fg.conf_int().loc["pct_FG"]
print(f"pct_FG: beta={b:.3f}, se={se:.3f}, p={p:.3f}, 95%CI=[{lo:.3f},{hi:.3f}]  -> 10pp effect={10*b:.3f}")
print(f"R² overall = {m_fg.rsquared:.3f} | R² adj = {m_fg.rsquared_adj:.3f}")

# 直接用 %FG 计算 within R²
r2_within_fg = within_r2(df["co2_per_1000p"], df["pct_FG"], "ladcode", "year")
print(f"R² within (FE) = {r2_within_fg:.3f}")

# ---------- 3) 结果解读模板（可选打印） ----------
sign = "positive" if b>0 else "negative"
print(f"\n[Interpretation] A 1 percentage-point increase in %FG is associated with a {sign} "
      f"change of {abs(b):.3f} tonnes of CO₂ per 1,000 people within LADs, "
      "controlling for LAD and year fixed effects (cluster-robust SEs by LAD).")



=== Model A1: Separate %F and %G (FE: LAD + Year, cluster by LAD) ===
                            OLS Regression Results                            
Dep. Variable:          co2_per_1000p   R-squared:                       0.990
Model:                            OLS   Adj. R-squared:                  0.989
Method:                 Least Squares   F-statistic:                 2.184e+04
Date:                Fri, 08 Aug 2025   Prob (F-statistic):               0.00
Time:                        17:06:40   Log-Likelihood:                -26224.
No. Observations:                4990   AIC:                         5.312e+04
Df Residuals:                    4655   BIC:                         5.530e+04
Df Model:                         334                                         
Covariance Type:              cluster                                         
                              coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------

/opt/conda/lib/python3.11/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 334, but rank is 17
  warnings.warn('covariance of constraints does not have full '
/tmp/ipykernel_121/2429444205.py:29: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  y_dm = y - y.groupby(g).transform("mean") - y.groupby(t).transform("mean") + y.mean()
/tmp/ipykernel_121/2429444205.py:30: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  x_dm = x - x.groupby(g).transform("mean") - x.groupby(t).transform("mean") + x.mean()



=== Model A2: Combined %FG (FE: LAD + Year, cluster by LAD) ===
                            OLS Regression Results                            
Dep. Variable:          co2_per_1000p   R-squared:                       0.989
Model:                            OLS   Adj. R-squared:                  0.989
Method:                 Least Squares   F-statistic:                 2.389e+04
Date:                Fri, 08 Aug 2025   Prob (F-statistic):               0.00
Time:                        17:06:41   Log-Likelihood:                -26273.
No. Observations:                4990   AIC:                         5.321e+04
Df Residuals:                    4656   BIC:                         5.539e+04
Df Model:                         333                                         
Covariance Type:              cluster                                         
                              coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------

/opt/conda/lib/python3.11/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 333, but rank is 16
  warnings.warn('covariance of constraints does not have full '
/tmp/ipykernel_121/2429444205.py:29: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  y_dm = y - y.groupby(g).transform("mean") - y.groupby(t).transform("mean") + y.mean()
/tmp/ipykernel_121/2429444205.py:30: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  x_dm = x - x.groupby(g).transform("mean") - x.groupby(t).transform("mean") + x.mean()


In [34]:
# === Urban vs. Suburban heterogeneity with FE & clustered SEs ===
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

# ---------- 0) 数据准备 ----------
# 需要的列：co2_per_1000p, pct_F/pct_G(或pct_FG), ladcode, year, urbanity(0/1)
need_any = {"co2_per_1000p", "ladcode", "year", "urbanity"}
if not need_any.issubset(set(panel.columns)):
    raise ValueError(f"缺少列：{need_any - set(panel.columns)}")

df = panel.copy()
# 若没有 pct_FG 就生成
if "pct_FG" not in df.columns:
    if not {"pct_F", "pct_G"}.issubset(df.columns):
        raise ValueError("缺少 pct_FG 且也没有 pct_F / pct_G，无法构造 %FG。")
    df["pct_FG"] = df["pct_F"].astype(float) + df["pct_G"].astype(float)

df = df.dropna(subset=["co2_per_1000p", "pct_FG", "ladcode", "year", "urbanity"]).copy()
df["year"] = df["year"].astype("int64")
df["urbanity"] = df["urbanity"].astype("int64")  # 0=郊区/非城市, 1=城市
df["ladcode"] = df["ladcode"].astype("category")

# ---------- 小工具：within R² ----------
def within_r2(data, ycol, xcol, unit="ladcode", time="year"):
    y = data[ycol].copy()
    x = data[xcol].copy()
    g = data[unit]
    t = data[time]
    y_dm = y - y.groupby(g).transform("mean") - y.groupby(t).transform("mean") + y.mean()
    x_dm = x - x.groupby(g).transform("mean") - x.groupby(t).transform("mean") + x.mean()
    m = sm.OLS(y_dm, sm.add_constant(x_dm)).fit()
    return float(m.rsquared)

# ============================================================
# 1) 交互项模型：co2_per_1000p ~ pct_FG * urbanity + LAD FE + Year FE
#    baseline(urbanity=0) 为郊区斜率；交互系数=城市与郊区斜率差
# ============================================================
formula_int = "co2_per_1000p ~ pct_FG*urbanity + C(ladcode) + C(year)"
m_int = smf.ols(formula_int, data=df).fit(
    cov_type="cluster", cov_kwds={"groups": df["ladcode"]}
)
print("\n=== Heterogeneity via interaction: %FG × urbanity ===")
print(m_int.summary())

# 提取并解释系数
b_base   = m_int.params.get("pct_FG", np.nan)                         # 郊区斜率
b_diff   = m_int.params.get("pct_FG:urbanity", np.nan)                # 城市-郊区 差异
se_base  = m_int.bse.get("pct_FG", np.nan)
se_diff  = m_int.bse.get("pct_FG:urbanity", np.nan)
p_base   = m_int.pvalues.get("pct_FG", np.nan)
p_diff   = m_int.pvalues.get("pct_FG:urbanity", np.nan)
lo_base, hi_base = m_int.conf_int().loc["pct_FG"]
lo_diff, hi_diff = m_int.conf_int().loc["pct_FG:urbanity"]

b_urban  = b_base + b_diff                                            # 城市斜率
# 近似城市斜率的标准误（忽略协方差项的近似；如需精确，用 delta method）
cov = m_int.cov_params()
se_urban = float(np.sqrt(
    cov.loc["pct_FG","pct_FG"] + cov.loc["pct_FG:urbanity","pct_FG:urbanity"]
    + 2*cov.loc["pct_FG","pct_FG:urbanity"]
)) if {"pct_FG","pct_FG:urbanity"}.issubset(cov.index) else np.nan

print(f"\n[Slopes]  郊区 slope = {b_base:.3f} (se={se_base:.3f}, p={p_base:.3f}, 95%CI=[{lo_base:.3f},{hi_base:.3f}])")
print(f"          城市-郊区 差 = {b_diff:.3f} (se={se_diff:.3f}, p={p_diff:.3f}, 95%CI=[{lo_diff:.3f},{hi_diff:.3f}])")
print(f"          城市 slope ≈ {b_urban:.3f} (se≈{se_urban:.3f})")
print(f"          10pp 效应：郊区 {10*b_base:.3f} 吨/1000人；城市 {10*b_urban:.3f} 吨/1000人")

print(f"R² overall = {m_int.rsquared:.3f} | R² adj = {m_int.rsquared_adj:.3f}")
# 用 %FG 计算 within R²（整样本）
print(f"R² within (FE, using %FG) = {within_r2(df, 'co2_per_1000p', 'pct_FG'):.3f}")

# ============================================================
# 2) 分组回归：城市组、郊区组分开估计（验证交互项结果）
# ============================================================
df_u = df[df["urbanity"]==1].copy()
df_s = df[df["urbanity"]==0].copy()

m_u = smf.ols("co2_per_1000p ~ pct_FG + C(ladcode) + C(year)", data=df_u)\
        .fit(cov_type="cluster", cov_kwds={"groups": df_u["ladcode"]})
m_s = smf.ols("co2_per_1000p ~ pct_FG + C(ladcode) + C(year)", data=df_s)\
        .fit(cov_type="cluster", cov_kwds={"groups": df_s["ladcode"]})

print("\n=== Split-sample FE regressions ===")
print("[Urban]");  print(m_u.summary())
print("[Suburban]"); print(m_s.summary())

# 摘要输出
def summarize(model, label):
    b  = model.params["pct_FG"]; se = model.bse["pct_FG"]; p = model.pvalues["pct_FG"]
    lo, hi = model.conf_int().loc["pct_FG"]
    print(f"{label}: beta={b:.3f}, se={se:.3f}, p={p:.3f}, 95%CI=[{lo:.3f},{hi:.3f}], 10pp={10*b:.3f}")
    print(f"R² overall={model.rsquared:.3f}  within={within_r2(model.model.data.frame, 'co2_per_1000p', 'pct_FG'):.3f}")

summarize(m_u, "Urban")
summarize(m_s, "Suburban")

# ============================================================
# 3) 可选稳健：滞后 + 交互（若你怀疑时滞）
# ============================================================
df = df.sort_values(["ladcode","year"]).copy()
df["pct_FG_l1"] = df.groupby("ladcode")["pct_FG"].shift(1)
df_lag = df.dropna(subset=["pct_FG_l1"]).copy()

m_int_lag = smf.ols("co2_per_1000p ~ pct_FG_l1*urbanity + C(ladcode) + C(year)", data=df_lag)\
             .fit(cov_type="cluster", cov_kwds={"groups": df_lag["ladcode"]})
print("\n=== Interaction with lagged %FG (t-1) ===")
print(m_int_lag.summary())



=== Heterogeneity via interaction: %FG × urbanity ===
                            OLS Regression Results                            
Dep. Variable:          co2_per_1000p   R-squared:                       0.990
Model:                            OLS   Adj. R-squared:                  0.989
Method:                 Least Squares   F-statistic:                 2.832e+04
Date:                Fri, 08 Aug 2025   Prob (F-statistic):               0.00
Time:                        17:31:25   Log-Likelihood:                -26077.
No. Observations:                4990   AIC:                         5.282e+04
Df Residuals:                    4655   BIC:                         5.501e+04
Df Model:                         334                                         
Covariance Type:              cluster                                         
                              coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------

/opt/conda/lib/python3.11/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 335, but rank is 17
  warnings.warn('covariance of constraints does not have full '
/tmp/ipykernel_121/1898961850.py:31: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  y_dm = y - y.groupby(g).transform("mean") - y.groupby(t).transform("mean") + y.mean()
/tmp/ipykernel_121/1898961850.py:32: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  x_dm = x - x.groupby(g).transform("mean") - x.groupby(t).transform("mean") + x.mean()



=== Split-sample FE regressions ===
[Urban]


/opt/conda/lib/python3.11/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 333, but rank is 16
  warnings.warn('covariance of constraints does not have full '


                            OLS Regression Results                            
Dep. Variable:          co2_per_1000p   R-squared:                       0.991
Model:                            OLS   Adj. R-squared:                  0.990
Method:                 Least Squares   F-statistic:                     8724.
Date:                Fri, 08 Aug 2025   Prob (F-statistic):          4.81e-310
Time:                        17:31:26   Log-Likelihood:                -18840.
No. Observations:                3658   AIC:                         3.817e+04
Df Residuals:                    3411   BIC:                         3.971e+04
Df Model:                         246                                         
Covariance Type:              cluster                                         
                              coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------
Intercept                2

/opt/conda/lib/python3.11/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 333, but rank is 16
  warnings.warn('covariance of constraints does not have full '
/tmp/ipykernel_121/1898961850.py:31: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  y_dm = y - y.groupby(g).transform("mean") - y.groupby(t).transform("mean") + y.mean()
/tmp/ipykernel_121/1898961850.py:32: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  x_dm = x - x.groupby(g).transform("mean") - x.groupby(t).transform("mean") + x.mean()
/tmp/ipykernel_121/189896185

                            OLS Regression Results                            
Dep. Variable:          co2_per_1000p   R-squared:                       0.993
Model:                            OLS   Adj. R-squared:                  0.992
Method:                 Least Squares   F-statistic:                 1.028e+04
Date:                Fri, 08 Aug 2025   Prob (F-statistic):          7.61e-134
Time:                        17:31:26   Log-Likelihood:                -6757.9
No. Observations:                1332   AIC:                         1.372e+04
Df Residuals:                    1229   BIC:                         1.426e+04
Df Model:                         102                                         
Covariance Type:              cluster                                         
                              coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------
Intercept                2

/opt/conda/lib/python3.11/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 334, but rank is 16
  warnings.warn('covariance of constraints does not have full '
